# Study 960 — The Unstaked ETF ⛓️

**A US spot-Ethereum ETF owns coins but does not stake them. What does that cost you?**

An on-chain holder can post their ether to the consensus layer and earn a protocol reward
of roughly 3% a year. Over most of this sample a US spot-ETH fund did not, so its holder
gave that reward up on top of the sponsor fee. The obvious test: measure how far each fund
fell behind the coin.

We run it on **ETHA** (0.25%/yr) and **ETHE** (2.50%/yr) against **ETH-USD**, daily closes,
2024-07-23 → 2026-06-30 (486 sessions). Then we find out that the obvious test
cannot work — and what does.

*Real numbers below are the frozen headline (`docs/results.md`, Fingerprint `4e34c697361e`).
The live cells run the offline synthetic control only. As-of 2026-06-30.*


## 1. The straightforward measurement — and why it says nothing

Take each fund's price, take the coin's price, and see how far apart they drift per year. That is a *tracking difference*, and it is how anyone would check whether a fund is quietly costing you 3% a year.

In [1]:
R = dict(etha_td=0.27, etha_lo=-8.86, etha_hi=9.34, etha_mt=0.11,
         ethe_td=-1.37, ethe_lo=-10.53, ethe_hi=7.77, ethe_mt=-0.56)
for name in ('etha', 'ethe'):
    print('%-5s vs ETH-USD: %+.2f%%/yr    95%% confidence [%+.2f, %+.2f]   monthly t = %+.2f'
          % (name.upper(), R[name+'_td'], R[name+'_lo'], R[name+'_hi'], R[name+'_mt']))

ETHA  vs ETH-USD: +0.27%/yr    95% confidence [-8.86, +9.34]   monthly t = +0.11
ETHE  vs ETH-USD: -1.37%/yr    95% confidence [-10.53, +7.77]   monthly t = -0.56


Look at the width of those brackets. The answer is *somewhere between minus nine and plus nine percent a year*. That is not a measurement, it is a shrug. The cause is timing: ether trades every hour of every day, and the price your data vendor stamps as the coin's daily close is taken several hours after the **16:00 New York** bell that closes the fund. On an asset that moves ~72% a year, those few hours dwarf everything. The day-to-day gap swings by **190 basis points** and then mostly reverses (-0.56 autocorrelation) — noise, not information.

> 🔬 **For the quants:** the resolution limit — half the confidence-interval width — is **not one number**. It depends on the ruler: ±18%/yr with 5-day bootstrap blocks, ±9.1 at 21, ±5.0 on non-overlapping months. We publish the sweep rather than the tidiest member of it. Any drag smaller than ±5.0%/yr is invisible on this tape however you hold the ruler — and the drag in question is ~3%/yr.

## 2. The deeper problem — the yardstick is unstaked too

Suppose the noise vanished. The measurement would *still* miss the staking yield, and this is the part almost everyone gets wrong.

The consensus reward is not paid into the price of a coin. It is paid **in extra coins**, to the validator who posted them. So `ETH-USD` — the market price of one ether — is the price of an **unstaked** coin. A fund that does not stake and a holder who does not stake are, on that yardstick, doing the same thing. The forgone reward cannot show up in the gap between them. It is not hiding in the noise; it is not in the data at all.

## 3. So where does the number come from? Two constants

Write the honest ledger. The tape gives one column; the other two are quoted from the issuer's fee schedule and from the Ethereum network — declared **assumptions**, not observations.

| | Measured vs the (unstaked) coin | Assumed fee | Assumed staking | Behind a self-staking holder |
|---|--:|--:|--:|--:|
| **ETHA** | +0.27%/yr | 0.25% | 3.00% | **-2.73%/yr** |
| **ETHE** | -1.37%/yr | 2.50% | 3.00% | **-4.37%/yr** |

On the cheap wrapper the forgone staking is **92%** of the whole declared cost of ownership — the fee is almost a rounding error next to it. On the dear one it is **55%**, because a 2.5% fee is finally big enough to compete. Both figures are true arithmetic. Neither is a finding: change the assumed yield and they change with it, and the tape does not vote.

## 4. What the same measurement *can* see

Now point the identical estimator at **ETHE against ETHA** — two wrappers on the same coin, both stamped at the same 16:00 bell. The timing error cancels exactly. The daily wobble collapses from 190 basis points to **11**, and a real number appears:

- **-1.64%/yr**, 95% confidence **[-2.27, -0.97]**, monthly *t* = **-5.16**.

The two funds' published fees differ by **2.25%/yr**, and that figure sits inside the interval. The expensive wrapper really does cost about what it says it costs — and you can prove it, because for once the benchmark shares the fund's closing bell. It holds up when you poke it: **22 of 24 months** are negative, and dropping any single month moves the answer only between -1.79 and -1.51%/yr.

> ⚠️ **Two things we are not hiding.** First, these are **2 of about 9 funds** that launched that day, and they are the cheapest and the dearest — the widest gap available (FETH, ETHW, ETHV, QETH, EZET, CETH are not measured here). Second, the crudest statistical test of all, which treats every day as independent, gives only *t* = -1.32. We do not use it, and notebook 02 §3 says exactly why.

## 5. Something changed halfway through

That spread is not constant. Cut the sample in July 2025 and it goes from **-2.43%/yr** to **-0.90%/yr**; year by year it reads **-2.14%** (2024), **-1.97%** (2025), **-0.52%** (2026).

A fee cut, a waiver rolling off, or one fund starting to stake its coins would all produce exactly this shape. **This tape cannot tell you which**, and pretending otherwise would be the very thing this study is complaining about.

## 6. Live check — the machinery is unbiased (offline synthetic)

*This cell runs on invented data, not the real tape.* We build one coin, a deliberately mis-timed vendor close, and two wrappers separated by a drag spread we choose ourselves. The fund-vs-fund estimator should find it; the fund-vs-coin estimator, fed the same world, should lose it completely.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from unstaked import data, strategy as st
px, truth = data.synthetic_panel(signal_strength=1.0, seed=960)
d = st.synthetic_detect(px)
print('we planted a spread of %.2f%%/yr between the two wrappers' % truth['spread_planted_pct'])
print('  fund vs fund : %+.2f%%/yr   95%% [%+.2f, %+.2f]  -> found it'
      % (d['fund_vs_fund_pct'], d['fund_vs_fund_ci'][0], d['fund_vs_fund_ci'][1]))
print('  fund vs coin : %+.2f%%/yr   95%% [%+.2f, %+.2f]  -> lost it (%.0fx wider)'
      % (d['fund_vs_coin_pct'], d['fund_vs_coin_ci'][0], d['fund_vs_coin_ci'][1],
         d['fund_vs_coin_halfwidth'] / d['fund_vs_fund_halfwidth']))
null = st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=960)[0])
print('  null (identical wrappers), fund vs fund: %+.3f%%/yr -> quiet, as it must be'
      % null['fund_vs_fund_pct'])

we planted a spread of 2.25%/yr between the two wrappers
  fund vs fund : -2.24%/yr   95% [-3.05, -1.45]  -> found it
  fund vs coin : +0.28%/yr   95% [-12.01, +12.95]  -> lost it (16x wider)


  null (identical wrappers), fund vs fund: +0.015%/yr -> quiet, as it must be


## Verdict

- **Signal — Mixed.** The staking gap the question asks about is **not visible** (+0.27%/yr, interval [-8.9, +9.3]) and never could be, because the coin benchmark is unstaked. What *is* visible is real and robust: the **-1.64%/yr** fee gap between the two wrappers, *t* = -5.16, matching 73% of the documented 2.25%/yr difference.
- **Tradability — Fragile.** There is one honest action: **own the cheap wrapper**. Worth the measured 1.64%/yr — 3.1% of your money over 1.92 years — against a *documented* fee gap of 2.25%/yr, for a single trade, no borrow, no cleverness. But it is 23 months of one asset and one fee-extreme pair, and it is already narrowing. The ~3%/yr staking prize is not available from inside any of these funds; collecting it means leaving the wrapper for self-custody or a staking provider, with the lock-up, slashing and tax questions that brings.
- **The lesson worth keeping.** Before you quote anyone a tracking difference, compute the confidence interval — then compute it a second way. Here it lands anywhere from ±5 to ±18%/yr depending on the method, and *every* version is bigger than the ~3%/yr everyone was arguing about. Quoting the single tidiest width would have been picking a ruler to fit the story.